## Введение 

В этом уроке рассмотрим: 
- Что такое вызов функции и случаи его использования 
- Как создать вызов функции с помощью Azure OpenAI 
- Как интегрировать вызов функции в приложение 

## Цели обучения 

После прохождения этого урока вы будете знать и понимать: 

- Цель использования вызова функции 
- Настройку вызова функции с использованием сервиса Azure Open AI 
- Проектирование эффективных вызовов функций для вашего сценария использования приложения 


## Понимание вызовов функций

Для этого урока мы хотим создать функцию для нашего образовательного стартапа, которая позволит пользователям использовать чат-бота для поиска технических курсов. Мы будем рекомендовать курсы, соответствующие их уровню навыков, текущей роли и интересующей технологии.

Для выполнения этого мы используем комбинацию:
 - `Azure Open AI` для создания опыта общения с пользователем в чате
 - `Microsoft Learn Catalog API` для помощи пользователям в поиске курсов на основе их запроса
 - `Function Calling`, чтобы взять запрос пользователя и отправить его функции для выполнения API-запроса.

Для начала давайте посмотрим, зачем нам вообще нужен вызов функций:


### Зачем нужен вызов функций

Если вы прошли любой другой урок в этом курсе, вы, вероятно, понимаете мощь использования больших языковых моделей (LLM). Надеюсь, вы также видите некоторые их ограничения.

Вызов функций — это функция сервиса Azure Open AI, которая предназначена для преодоления следующих ограничений:
1) Последовательный формат ответа
2) Возможность использовать данные из других источников приложения в контексте чата

До появления вызова функций ответы от LLM были неструктурированными и непоследовательными. Разработчикам приходилось писать сложный код проверки, чтобы правильно обрабатывать каждое вариации ответа.

Пользователи не могли получить ответы на такие вопросы, как «Какая сейчас погода в Стокгольме?». Это связано с тем, что модели ограничивались временем, на котором обучались данные.

Рассмотрим следующий пример, иллюстрирующий эту проблему:

Допустим, мы хотим создать базу данных студентов, чтобы предлагать им подходящие курсы. Ниже приведены два описания студентов, которые очень похожи по содержанию данных.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Мы хотим отправить это в LLM для разбора данных. Это позже может быть использовано в нашем приложении для отправки на API или сохранения в базе данных.

Давайте создадим два идентичных запроса, в которых мы укажем LLM, какую информацию нас интересует:


Мы хотим отправить это в LLM, чтобы разобрать части, важные для нашего продукта. Таким образом мы можем создать два идентичных запроса для инструкции LLM:


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


После создания этих двух запросов мы отправим их в LLM, используя `client.responses.create`. Мы сохраняем запрос в переменную `input` и назначаем роль `user`. Это имитирует сообщение пользователя, отправляемое в чат-бот. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Теперь мы можем отправить оба запроса в LLM и изучить полученный ответ. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Несмотря на то, что подсказки одинаковы и описания похожи, мы можем получить разные форматы свойства `Grades`. 

Если запустить приведённую выше ячейку несколько раз, формат может быть как `3.7`, так и `3.7 GPA`. 

Это происходит потому, что LLM принимает неструктурированные данные в виде написанной подсказки и возвращает тоже неструктурированные данные. Нам нужен структурированный формат, чтобы мы знали, чего ожидать при сохранении или использовании этих данных.

Используя функциональный вызов, мы можем гарантировать получение обратно структурированных данных. При использовании функционального вызова LLM фактически не вызывает и не выполняет никаких функций. Вместо этого мы создаём структуру, которой LLM должен следовать в своих ответах. Затем мы используем эти структурированные ответы, чтобы понять, какую функцию запускать в наших приложениях.  
 


![Диаграмма вызова функции](../../../../translated_images/ru/Function-Flow.083875364af4f4bb.webp)


Затем мы можем взять то, что возвращает функция, и отправить это обратно в LLM. LLM затем ответит, используя естественный язык, чтобы ответить на запрос пользователя. 


### Случаи использования вызовов функций 

**Вызов внешних инструментов** 
Чат-боты отлично подходят для предоставления ответов на вопросы пользователей. Используя вызовы функций, чат-боты могут использовать сообщения пользователей для выполнения определённых задач. Например, студент может попросить чат-бота "Отправить электронное письмо моему преподавателю с просьбой о дополнительной помощи по этому предмету". Это может инициировать вызов функции `send_email(to: string, body: string)`


**Создание запросов к API или базе данных**
Пользователи могут искать информацию, используя естественный язык, который преобразуется в форматированный запрос или API-запрос. Примером может быть учитель, который запрашивает "Кто выполнил последнее задание", что может вызвать функцию с именем `get_completed(student_name: string, assignment: int, current_status: string)`


**Создание структурированных данных**
Пользователи могут взять блок текста или CSV и использовать LLM для извлечения важной информации из него. Например, студент может преобразовать статью из Википедии о мирных соглашениях для создания AI-флешкарт. Это можно сделать, используя функцию `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Создание вашего первого вызова функции 

Процесс создания вызова функции включает 3 основных шага: 
1. Вызов API Chat Completions с списком ваших функций и сообщением от пользователя 
2. Чтение ответа модели для выполнения действия, то есть выполнение функции или вызова API 
3. Выполнение ещё одного вызова к API Chat Completions с ответом вашей функции, чтобы использовать эту информацию для создания ответа пользователю. 


![Поток вызова функции](../../../../translated_images/ru/LLM-Flow.3285ed8caf4796d7.webp)


### Элементы вызова функции

#### Ввод пользователя

Первый шаг — создать сообщение пользователя. Это может быть значение, динамически присвоенное из текстового ввода, или вы можете присвоить значение здесь. Если вы впервые работаете с API Chat Completions, нам нужно определить `role` и `content` сообщения.

`role` может быть `system` (создающий правила), `assistant` (модель) или `user` (конечный пользователь). Для вызова функции мы назначим это значение `user` и пример вопроса.


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Создание функций. 

Далее мы определим функцию и параметры этой функции. Здесь мы используем только одну функцию под названием `search_courses`, но вы можете создать несколько функций.

**Важно** : Функции включаются в системное сообщение для LLM и будут учитываться в количестве доступных вам токенов. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Определения** 

`name` - Имя функции, которую мы хотим вызвать. 

`description` - Это описание того, как работает функция. Здесь важно быть конкретным и понятным 

`parameters` - Список значений и формата, которые вы хотите, чтобы модель создавала в своём ответе 


`type` - Тип данных, в котором будут храниться свойства 

`properties` - Список конкретных значений, которые модель будет использовать для своего ответа 


`name` - имя свойства, которое модель будет использовать в отформатированном ответе 

`type` - Тип данных этого свойства 

`description` - Описание конкретного свойства 


**Необязательно**

`required` - обязательное свойство для завершения вызова функции 


### Вызов функции 
После определения функции нам теперь нужно включить её в вызов Chat Completion API. Для этого мы добавляем `functions` в запрос. В данном случае `functions=functions`. 

Также есть опция установить `function_call` в значение `auto`. Это означает, что мы позволим модели LLM самостоятельно решить, какую функцию вызывать, исходя из сообщения пользователя, а не назначать это сами.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Теперь давайте посмотрим на ответ и увидим, как он отформатирован: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Вы можете видеть, что вызывается имя функции, и из сообщения пользователя LLM смог найти данные, подходящие к аргументам функции. 


## 3. Интеграция вызовов функций в приложение. 


После того как мы протестировали отформатированный ответ от LLM, теперь мы можем интегрировать это в приложение. 

### Управление потоком 

Чтобы интегрировать это в наше приложение, давайте выполним следующие шаги: 

Сначала сделаем вызов сервисов Open AI и сохраним сообщение в переменную с именем `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Теперь мы определим функцию, которая будет вызывать API Microsoft Learn для получения списка курсов: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



В качестве лучшей практики мы затем проверим, хочет ли модель вызвать функцию. После этого мы создадим одну из доступных функций и сопоставим ее с вызываемой функцией. 
Затем мы возьмем аргументы функции и сопоставим их с аргументами из LLM.

Наконец, мы добавим сообщение о вызове функции и значения, которые были возвращены сообщением `search_courses`. Это дает LLM всю необходимую информацию для
ответа пользователю на естественном языке. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Теперь мы отправим обновленное сообщение в LLM, чтобы получить ответ на естественном языке вместо ответа в формате API JSON. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Кодовое испытание 

Отличная работа! Чтобы продолжить обучение вызову функций Azure Open AI, вы можете создать: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Больше параметров функции, которые могут помочь учащимся найти больше курсов. Доступные параметры API можно найти здесь: 
 - Создайте другой вызов функции, который принимает больше информации от учащегося, например, их родной язык 
 - Создайте обработку ошибок на случай, если вызов функции и/или вызов API не вернут подходящие курсы 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
